notebook for full loading all the data through api 

In [0]:
import calendar
import json
import time
from datetime import datetime, timezone
import requests

# ── Configuration ────────────────────────────────────────────────────────────
SA_NAME = "stmaritimeks"
BRONZE_ROOT = f"abfss://bronze@{SA_NAME}.dfs.core.windows.net"
BASE_URL = "https://gateway.api.globalfishingwatch.org/v3"

# Key Vault scope & secret configured by Member 1
KV_SCOPE = "maritime-kv"
API_TOKEN = dbutils.secrets.get(scope=KV_SCOPE, key="gfw-api-token")

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}

# Initial load window (matches project historical scope)
START_MONTH = "2025-01"
END_MONTH = "2025-02"
MAX_ROWS_PER_CHUNK = 15000

WORLD_DATASETS = {
    "gap": "public-global-gaps-events:latest",
    "fishing": "public-global-fishing-events:latest",
    "loitering": "public-global-loitering-events:latest",
    "encounter": "public-global-encounters-events:latest",
}

PORT_VISIT_DATASET = "public-global-port-visits-events:latest"

# 13 Major Global Ports bounding boxes [min_lon, min_lat, max_lon, max_lat]
PORT_BOXES = {
    "shanghai": [121.30, 30.50, 122.30, 31.50],
    "singapore": [103.55, 1.10, 104.10, 1.40],
    "busan": [128.90, 34.95, 129.20, 35.20],
    "hong_kong": [113.80, 22.15, 114.35, 22.45],
    "rotterdam": [3.95, 51.85, 4.55, 52.05],
    "antwerp": [4.15, 51.20, 4.45, 51.40],
    "hamburg": [9.75, 53.45, 10.05, 53.60],
    "la_longbeach": [-118.35, 33.68, -118.10, 33.82],
    "new_york_nj": [-74.22, 40.55, -73.95, 40.75],
    "jebel_ali": [54.95, 24.90, 55.15, 25.08],
    "port_klang": [101.25, 2.90, 101.45, 3.10],
    "santos": [-46.40, -24.05, -46.25, -23.90],
    "karachi": [66.85, 24.72, 67.05, 24.88],
}


def month_windows(start_ym: str, end_ym: str):
    sy, sm = map(int, start_ym.split("-"))
    ey, em = map(int, end_ym.split("-"))
    y, m = sy, sm
    windows = []
    while (y, m) <= (ey, em):
        last_day = calendar.monthrange(y, m)[1]
        ym_str = f"{y:04d}-{m:02d}"
        windows.append((ym_str, f"{ym_str}-01", f"{ym_str}-{last_day:02d}"))
        if m == 12:
            y += 1
            m = 1
        else:
            m += 1
    return windows


def bbox_to_polygon(bbox):
    min_lon, min_lat, max_lon, max_lat = bbox
    return {
        "type": "Polygon",
        "coordinates": [[
            [min_lon, min_lat],
            [max_lon, min_lat],
            [max_lon, max_lat],
            [min_lon, max_lat],
            [min_lon, min_lat],
        ]],
    }


def fetch_events_with_retry(dataset: str, start_date: str, end_date: str, geometry=None, max_rows=15000):
    url = f"{BASE_URL}/events"
    limit = 1000
    offset = 0
    collected = []
    retries_used = 0

    while len(collected) < max_rows:
        params = {"limit": min(limit, max_rows - len(collected)), "offset": offset}
        payload = {
            "datasets": [dataset],
            "startDate": f"{start_date}T00:00:00.000Z",
            "endDate": f"{end_date}T23:59:59.999Z",
        }
        if geometry is not None:
            payload["geometry"] = geometry

        for attempt in range(1, 6):
            resp = requests.post(url, headers=HEADERS, params=params, json=payload, timeout=60)
            if resp.status_code == 200:
                break
            if resp.status_code in (429, 500, 502, 503, 504):
                retries_used += 1
                wait_sec = 2 ** attempt
                print(f"  [HTTP {resp.status_code}] Retrying in {wait_sec}s (attempt {attempt}/5)...")
                time.sleep(wait_sec)
            else:
                resp.raise_for_status()
        else:
            raise RuntimeError(f"Failed after 5 retries for {dataset} {start_date}..{end_date}")

        data = resp.json()
        entries = data.get("entries", [])
        if not entries:
            break

        collected.extend(entries)
        if len(entries) < params["limit"]:
            break
        offset += len(entries)
        time.sleep(0.25)

    return collected, retries_used


def run_initial_ingestion():
    ingest_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    run_id = f"initial_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    windows = month_windows(START_MONTH, END_MONTH)
    manifest_records = []

    # 1. Global events (gap, fishing, loitering, encounter)
    for event_type, dataset in WORLD_DATASETS.items():
        for ym_str, win_start, win_end in windows:
            started_at = datetime.now(timezone.utc).isoformat()
            target_path = (
                f"{BRONZE_ROOT}/gfw/events/{event_type}/load_type=initial/"
                f"ingest_date={ingest_date}/world_{ym_str}.jsonl"
            )
            try:
                rows, retries = fetch_events_with_retry(
                    dataset=dataset,
                    start_date=win_start,
                    end_date=win_end,
                    geometry=None,
                    max_rows=MAX_ROWS_PER_CHUNK,
                )
                body = "\n".join(json.dumps(r) for r in rows)
                dbutils.fs.put(target_path, body, True)
                status = "SUCCESS"
                err_msg = None
                print(f"[OK] {event_type} world {ym_str}: {len(rows)} rows -> {target_path}")
            except Exception as exc:
                rows = []
                retries = 5
                status = "FAILED"
                err_msg = str(exc)
                print(f"[ERR] {event_type} world {ym_str}: {err_msg}")

            manifest_records.append({
                "run_id": run_id,
                "entity": event_type,
                "dataset": dataset,
                "region": "world",
                "window_start": win_start,
                "window_end": win_end,
                "load_type": "initial",
                "source": "gfw_v3_events_api",
                "rows_received": len(rows),
                "file_path": target_path,
                "status": status,
                "retry_count": retries,
                "error_message": err_msg,
                "started_at_utc": started_at,
                "finished_at_utc": datetime.now(timezone.utc).isoformat(),
                "source_url": f"{BASE_URL}/events",
                "api_version": "v3",
            })

    # 2. Port visits across 13 major ports
    for port_name, bbox in PORT_BOXES.items():
        geom = bbox_to_polygon(bbox)
        for ym_str, win_start, win_end in windows:
            started_at = datetime.now(timezone.utc).isoformat()
            target_path = (
                f"{BRONZE_ROOT}/gfw/events/port_visit/load_type=initial/"
                f"ingest_date={ingest_date}/{port_name}_{ym_str}.jsonl"
            )
            try:
                rows, retries = fetch_events_with_retry(
                    dataset=PORT_VISIT_DATASET,
                    start_date=win_start,
                    end_date=win_end,
                    geometry=geom,
                    max_rows=50000,
                )
                body = "\n".join(json.dumps(r) for r in rows)
                dbutils.fs.put(target_path, body, True)
                status = "SUCCESS"
                err_msg = None
                print(f"[OK] port_visit {port_name} {ym_str}: {len(rows)} rows -> {target_path}")
            except Exception as exc:
                rows = []
                retries = 5
                status = "FAILED"
                err_msg = str(exc)
                print(f"[ERR] port_visit {port_name} {ym_str}: {err_msg}")

            manifest_records.append({
                "run_id": run_id,
                "entity": "port_visit",
                "dataset": PORT_VISIT_DATASET,
                "region": port_name,
                "window_start": win_start,
                "window_end": win_end,
                "load_type": "initial",
                "source": "gfw_v3_events_api",
                "rows_received": len(rows),
                "file_path": target_path,
                "status": status,
                "retry_count": retries,
                "error_message": err_msg,
                "started_at_utc": started_at,
                "finished_at_utc": datetime.now(timezone.utc).isoformat(),
                "source_url": f"{BASE_URL}/events",
                "api_version": "v3",
            })

    manifest_path = f"{BRONZE_ROOT}/_manifest/ingestion_manifest/{run_id}.jsonl"
    dbutils.fs.put(manifest_path, "\n".join(json.dumps(r) for r in manifest_records), True)
    print(f"\nManifest written to {manifest_path}")

    watermark_path = f"{BRONZE_ROOT}/_state/watermark.json"
    last_end = windows[-1][2]
    dbutils.fs.put(
        watermark_path,
        json.dumps({
            "last_loaded_month": END_MONTH,
            "window_end": last_end,
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
            "run_id": run_id,
        }, indent=2),
        True,
    )
    print(f"Watermark updated to {END_MONTH} ({last_end})")


if __name__ == "__main__":
    run_initial_ingestion()